In [ ]:
import random
import shutil
from pathlib import Path

BASE_DIR = Path.cwd().parent

def split_and_copy(dataset_folder, ratios=(0.8, 0.1, 0.1), seed=42):
    print(f"=== Memproses Dataset: {dataset_folder} ===")
    random.seed(seed)

    # Path diarahkan berdasarkan BASE_DIR
    src_images = BASE_DIR / "data" / dataset_folder / "images"
    src_labels = BASE_DIR / "data" / dataset_folder / "labels"
    
    # Folder tujuan juga diletakkan di BASE_DIR
    dest_root = BASE_DIR / "dataset" / dataset_folder

    if not src_images.exists() or not src_labels.exists():
        print(f"❌ Error: Folder sumber tidak ditemukan di {src_images}")
        print("-" * 40)
        return

    valid_extensions = {'.jpg', '.jpeg', '.png'}
    images = [f for f in src_images.iterdir() if f.suffix.lower() in valid_extensions]
    
    valid_data = []
    for img in images:
        label_file = src_labels / f"{img.stem}.txt"
        if label_file.exists():
            valid_data.append((img, label_file))
    
    random.shuffle(valid_data)

    total = len(valid_data)
    train_end = int(total * ratios[0])
    val_end = train_end + int(total * ratios[1])

    splits = {
        'train': valid_data[:train_end],
        'val': valid_data[train_end:val_end],
        'test': valid_data[val_end:]
    }

    print(f"Total data valid: {total} pasang gambar & label.")

    for split_name, files in splits.items():
        dest_img_dir = dest_root / split_name / 'images'
        dest_lbl_dir = dest_root / split_name / 'labels'
        
        dest_img_dir.mkdir(parents=True, exist_ok=True)
        dest_lbl_dir.mkdir(parents=True, exist_ok=True)

        for img_path, lbl_path in files:
            shutil.copy(img_path, dest_img_dir / img_path.name)
            shutil.copy(lbl_path, dest_lbl_dir / lbl_path.name)
        
        print(f"✅ [{split_name.upper()}] Menyalin {len(files)} file ke {dest_img_dir.relative_to(BASE_DIR)}")
    print("-" * 40)

# EKSEKUSI
split_and_copy("mwpd")
split_and_copy("n-rdd2024")

=== Memproses Dataset: n-rdd2024 ===
Total data valid: 2376 pasang gambar & label.
✅ [TRAIN] Menyalin 1900 file ke dataset/n-rdd2024/train/images
✅ [VAL] Menyalin 237 file ke dataset/n-rdd2024/val/images
✅ [TEST] Menyalin 239 file ke dataset/n-rdd2024/test/images
----------------------------------------
